# Argoverse 1 Motion Forecasting — Exploration

This notebook loads one scenario from the Argoverse 1 Motion Forecasting dataset (via Kaggle), visualizes trajectories, and runs the preprocessing pipeline. Designed to run in **Google Colab** or locally.

## 1. Setup (Colab: add project to path and install deps)

In [ ]:
# Clone project (Colab: run this first). If repo exists, clean cache and pull branch.
import os
import sys

REPO_URL = "https://github.com/PulockDas/Implement-STAST-System.git"  # set your repo URL
BRANCH = "master"  # branch to checkout
CLONE_DIR = "/content/Implement-STAST-System"  # Colab; use os.getcwd() for local

ip = get_ipython()
if os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    ip.run_line_magic("cd", CLONE_DIR)
    ip.system("git fetch origin")
    ip.system("git clean -fd")
    ip.system(f"git checkout {BRANCH}")
    ip.system(f"git pull origin {BRANCH}")
    print(f"Updated existing repo at {CLONE_DIR} (branch {BRANCH})")
else:
    parent = os.path.dirname(CLONE_DIR) or "."
    os.makedirs(parent, exist_ok=True)
    ip.run_line_magic("cd", parent)
    ip.system(f"git clone --branch {BRANCH} {REPO_URL} {os.path.basename(CLONE_DIR)}")
    print(f"Cloned repo to {CLONE_DIR} (branch {BRANCH})")

PROJECT_ROOT = CLONE_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
ip.run_line_magic("cd", PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# If you didn't run the clone cell (e.g. local): set project root to parent of notebooks/.
import sys
import os

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# Install dependencies (skip if already installed, e.g. in Colab)
!pip install -q tqdm

## 2. Download dataset with kagglehub

In [ ]:
import kagglehub

# Download Argoverse 1 Motion Dataset from Kaggle
path = kagglehub.dataset_download("narendarmallireddy/argoverse1-motion-dataset")
print("Dataset path:", path)

In [ ]:
# Find scenario CSV files (layout may vary: train/val/test or flat)
from pathlib import Path

data_path = Path(path)
csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")
if csv_files:
    # Use first CSV as example
    scenario_path = str(csv_files[0])
    print("Example scenario:", scenario_path)
else:
    raise FileNotFoundError("No CSV files found under dataset path. Check dataset layout.")

## 3. Load one scenario

In [ ]:
from datasets.argoverse_loader import load_scenario_from_path

scenario = load_scenario_from_path(scenario_path, include_city=True)
trajectories = scenario["trajectories"]
print(f"Number of tracks: {len(trajectories)}")
for i, t in enumerate(trajectories[:3]):
    print(f"  Track {i}: track_id={t['track_id']}, object_type={t['object_type']}, "
          f"len(trajectory)={len(t['trajectory'])}, len(timestamps)={len(t['timestamps'])}")

In [ ]:
# Show one trajectory structure (output format)
example = trajectories[0]
print("Example trajectory dict keys:", list(example.keys()))
print("First 3 points (x,y):", example["trajectory"][:3])
print("First 3 timestamps:", example["timestamps"][:3])

## 4. Visualize trajectories (past vs future, highlight target)

In [ ]:
import matplotlib.pyplot as plt
from visualization.plot_trajectories import plot_scenario_trajectories
from utils.config import PAST_STEPS, FUTURE_STEPS

# Highlight AGENT if present, else first track
target_id = None
for t in trajectories:
    if t.get("object_type") == "AGENT":
        target_id = t["track_id"]
        break
if target_id is None and trajectories:
    target_id = trajectories[0]["track_id"]

fig = plot_scenario_trajectories(
    trajectories,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    target_track_id=target_id,
    title="Argoverse scenario — past (blue) vs future (coral), target in bold"
)
plt.show()

## 5. Preprocessing: past/future extraction and PyTorch tensors

In [ ]:
from preprocessing.trajectory_processor import prepare_agent_tensors
from utils.config import PAST_STEPS, FUTURE_STEPS, FEATURE_DIM_POSITION, FEATURE_DIM_WITH_VELOCITY

out = prepare_agent_tensors(
    trajectories,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    add_velocity=True,
    device=None,
)

past_t = out["past_tensor"]
future_t = out["future_tensor"]
print("past_tensor shape:", past_t.shape)
print("future_tensor shape:", future_t.shape)
print("Expected: past [num_agents, past_steps, features], future [num_agents, future_steps, 2]")
print("Features: past includes (x,y,vx,vy) ->", FEATURE_DIM_WITH_VELOCITY, "; future (x,y) ->", FEATURE_DIM_POSITION)
print("Track IDs:", out["track_ids"])
print("Object types:", out["object_types"])

In [ ]:
# Quick sanity check: first agent past positions vs raw trajectory
import numpy as np
first_past = out["past_list"][0]
first_traj = [t for t in trajectories if t["track_id"] == out["track_ids"][0]][0]["trajectory"]
np.testing.assert_allclose(first_past[:, :2], np.array(first_traj[:PAST_STEPS]), err_msg="Past positions should match")
print("Sanity check passed: past positions match raw trajectory.")

## 6. LSTM baseline (encoder–decoder)

Predict 30 future steps from 20 past steps.

- Input: `(batch, 20, 2)`
- Output: `(batch, 30, 2)`

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets.argoverse_dataset import ArgoverseSceneDataset
from models.lstm_baseline import LSTMTrajectoryPredictor
from metrics.trajectory_metrics import ade, fde
from utils.config import PAST_STEPS, FUTURE_STEPS

# Build scene-level dataset from multiple CSVs
max_scenes = 64  # limit for Colab
scene_csvs = csv_files[:max_scenes]
scene_dataset = ArgoverseSceneDataset(
    scene_csvs,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    max_vehicles=20,
    distance_threshold=50.0,
)
loader = DataLoader(scene_dataset, batch_size=16, shuffle=True, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMTrajectoryPredictor(input_dim=2, hidden_size=128).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Number of scenes:", len(scene_dataset))
print("Device:", device)

In [ ]:
# Quick training loop (baseline sanity check) using scene-level batches
model.train()
EPOCHS = 5

for epoch in range(EPOCHS):
    total_loss = 0.0
    total_ade = 0.0
    total_fde = 0.0
    n_batches = 0

    for batch in loader:
        # batch["past_traj"]: (B, max_vehicles, past_steps, 2)
        # batch["future_target"]: (B, future_steps, 2)
        # batch["vehicle_mask"]: (B, max_vehicles)
        past_scene = batch["past_traj"].to(device)
        future_target = batch["future_target"].to(device)

        # Backward-compat LSTM: use only target vehicle (index 0)
        past_target = past_scene[:, 0, :, :]  # (B, 20, 2)

        optimizer.zero_grad()
        pred = model(past_target)
        loss = criterion(pred, future_target)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            total_loss += loss.item()
            total_ade += ade(pred, future_target).item()
            total_fde += fde(pred, future_target).item()
            n_batches += 1

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"MSE: {total_loss/max(n_batches,1):.6f} | "
        f"ADE: {total_ade/max(n_batches,1):.4f} | "
        f"FDE: {total_fde/max(n_batches,1):.4f}"
    )

In [ ]:
# Simple DataLoader + model shape checks
model.eval()

batch = next(iter(loader))
past_scene = batch["past_traj"]          # (B, max_vehicles, 20, 2)
future_target = batch["future_target"]    # (B, 30, 2)
vehicle_mask = batch["vehicle_mask"]      # (B, max_vehicles)

print("past_traj shape:", tuple(past_scene.shape))
print("future_target shape:", tuple(future_target.shape))
print("vehicle_mask shape:", tuple(vehicle_mask.shape))

with torch.no_grad():
    pred = model(past_scene[:, 0, :, :].to(device)).cpu()

print("pred shape:", tuple(pred.shape))
print("expected:", (past_scene.shape[0], FUTURE_STEPS, 2))

## 7. Summary

- **Dataset loader**: `load_scenario_from_path(csv_path)` returns trajectories grouped by `TRACK_ID` with `trajectory`, `timestamps`, `object_type`.
- **Preprocessing**: `prepare_agent_tensors(trajectories)` yields `past_tensor` and `future_tensor` ready for PyTorch models.
- **Baseline model (LSTM)**: `LSTMTrajectoryPredictor` predicts `(batch, 30, 2)` from `(batch, 20, 2)`.
- **Visualization**: `plot_scenario_trajectories(trajectories, target_track_id=...)` plots past vs future and highlights the target vehicle.